In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

num_samples = 2000

data = {
    "ph": np.random.rand(num_samples),
    "Hardness": np.random.rand(num_samples),
    "Solids": np.random.rand(num_samples),
    "Chloramines": np.random.rand(num_samples),
    "Sulfate": np.random.rand(num_samples),
    "Conductivity": np.random.rand(num_samples),
    "Organic_carbon": np.random.rand(num_samples),
    "Trihalomethanes": np.random.rand(num_samples),
    "Turbidity": np.random.rand(num_samples),
}

df = pd.DataFrame(data)

# Create a binary target with some logic + noise
df["Potability"] = (
    (df["ph"] > 0.4) &
    (df["Hardness"] > 0.3) &
    (df["Solids"] < 0.7) &
    (df["Turbidity"] < 0.6)
).astype(int)

df.head()


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,0.374540,0.261706,0.571996,0.648257,0.720268,0.373641,0.654306,0.073175,0.004402,0
1,0.950714,0.246979,0.805432,0.172386,0.687283,0.332912,0.080033,0.089478,0.000330,0
2,0.731994,0.906255,0.760161,0.872395,0.095754,0.176154,0.242330,0.651974,0.472263,0
3,0.598658,0.249546,0.153900,0.613116,0.922572,0.607267,0.773679,0.486941,0.029294,0
4,0.156019,0.271950,0.149249,0.157204,0.568472,0.476624,0.528686,0.790415,0.974533,0


In [ ]:
dataset = df.to_csv('water_potability.csv')

In [ ]:
from torch.utils.data import Dataset,DataLoader
from torch.utils.data import random_split


class WaterDataset(Dataset):
  def __init__(self,csv_path):
    super().__init__()
    df = pd.read_csv(csv_path, index_col=0) # Add index_col=0 to ignore the DataFrame index as a feature
    self.data = df.to_numpy()
  def __len__(self):
    return len(self.data)
  def __getitem__(self,idx):
    features = self.data[idx,:-1]
    label = self.data[idx,-1]
    return features,label

dataset = WaterDataset("/content/water_potability.csv")
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset, [train_size, test_size]
)

dataloader_train = DataLoader(train_dataset,batch_size =2,shuffle =True)

features,label = next(iter(dataloader_train))
features,label

(tensor([[0.1834, 0.4111, 0.2598, 0.7535, 0.7722, 0.0141, 0.6243, 0.2087, 0.5448],
         [0.5117, 0.0044, 0.8024, 0.6658, 0.0348, 0.2199, 0.0642, 0.1907, 0.4892]],
        dtype=torch.float64),
 tensor([0., 0.], dtype=torch.float64))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional

In [ ]:
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(9,16)
    self.fc2 = nn.Linear(16,8)
    self.fc3 = nn.Linear(8,1)
  def forward(self,X):
    X = nn.functional.relu(self.fc1(X))
    X = nn.functional.relu(self.fc2(X))
    X = nn.functional.sigmoid(self.fc3(X))
    return X

In [ ]:
net = Net()

In [ ]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.SGD(net.parameters(),lr =0.01)

for epoch in range(1000):
  for features,labels in dataloader_train:
    # Cast features to torch.float32 to match model's expected input type
    features = features.to(torch.float32)
    # Cast labels to torch.float32 to match criterion's expected input type
    labels = labels.to(torch.float32)
    optimizer.zero_grad()
    outputs = net(features)
    loss = criterion(outputs,labels.view(-1,1))
    loss.backward()
    optimizer.step()

In [ ]:
dataloader_test = DataLoader(test_dataset,batch_size =2,shuffle =True)

features,label = next(iter(dataloader_test))
features,label

(tensor([[4.9588e-01, 8.2298e-01, 5.1474e-01, 4.6087e-01, 7.4613e-01, 2.8815e-01,
          1.8248e-01, 9.8487e-01, 2.2298e-01],
         [9.1093e-01, 8.5396e-01, 7.3461e-01, 7.7147e-01, 4.1029e-04, 4.5300e-01,
          6.6540e-01, 8.8528e-01, 5.9211e-01]], dtype=torch.float64),
 tensor([1., 0.], dtype=torch.float64))

In [ ]:
from torchmetrics import Accuracy

acc = Accuracy(task = "binary")
net.eval()

with torch.no_grad():
  for features,labels in dataloader_test:
    # Cast features to torch.float32 to match model's expected input type
    features = features.to(torch.float32)
    outputs = net(features)
    preds = (outputs>0.5).float()
    acc(preds,labels.view(-1,1))
accuracy = acc.compute()

In [ ]:
accuracy

tensor(0.9825)